# Demo 2 Setup

This notebook creates the **Bronze**, **Silver**, and **Gold** schemas and tables used in **Demo 2: Introduction to the Medallion Architecture**.

It simulates a realistic medallion flow using data derived from the `samples.tpch` catalog.

In [0]:
# Get current user and set up schema names
import re

username = spark.sql("SELECT current_user()").first()[0]
clean_username = re.sub(r'[^a-zA-Z0-9]', '_', username.split('@')[0])
catalog_name = spark.sql("SELECT current_catalog()").first()[0]

# Create three schemas representing the medallion layers
bronze_schema = f"demo_medallion_{clean_username}_bronze"
silver_schema = f"demo_medallion_{clean_username}_silver"
gold_schema = f"demo_medallion_{clean_username}_gold"

for schema in [bronze_schema, silver_schema, gold_schema]:
    layer = schema.split('_')[-1]
    spark.sql(f"CREATE SCHEMA IF NOT EXISTS `{catalog_name}`.`{schema}` COMMENT '{layer.capitalize()} layer - Medallion Architecture Demo'")

print(f"\u2713 Created medallion schemas in catalog: {catalog_name}")
print(f"  Bronze: {catalog_name}.{bronze_schema}")
print(f"  Silver: {catalog_name}.{silver_schema}")
print(f"  Gold:   {catalog_name}.{gold_schema}")

In [0]:
# ============================================================
# BRONZE LAYER: Raw data with intentional quality issues
# ============================================================
# Bronze simulates raw data landing exactly as it arrives from
# source systems — with duplicates, nulls, and raw codes.

spark.sql(f"USE SCHEMA `{bronze_schema}`")

# Create raw_orders: includes duplicate rows and some null values
spark.sql("""
CREATE OR REPLACE TABLE raw_orders
COMMENT 'Raw orders ingested from source system - contains duplicates and quality issues'
AS
SELECT 
  o_orderkey,
  o_custkey,
  o_orderstatus,
  o_totalprice,
  o_orderdate,
  o_orderpriority,
  o_clerk,
  o_shippriority,
  current_timestamp() AS _ingested_at
FROM samples.tpch.orders
WHERE o_orderdate >= '1995-01-01'

UNION ALL

-- Simulate duplicates: re-ingest ~5% of records (as happens with at-least-once delivery)
SELECT 
  o_orderkey,
  o_custkey,
  o_orderstatus,
  o_totalprice,
  o_orderdate,
  o_orderpriority,
  o_clerk,
  o_shippriority,
  current_timestamp() AS _ingested_at
FROM samples.tpch.orders
WHERE o_orderdate >= '1995-01-01' AND o_orderkey % 20 = 0
""")

# Create raw_customers: uses numeric nation keys (not human-readable names)
spark.sql("""
CREATE OR REPLACE TABLE raw_customers
COMMENT 'Raw customer data from source system - contains numeric codes instead of readable values'
AS
SELECT 
  c_custkey,
  c_name,
  c_address,
  c_nationkey,
  c_phone,
  c_acctbal,
  c_mktsegment,
  c_comment,
  current_timestamp() AS _ingested_at
FROM samples.tpch.customer
""")

print(f"\u2713 Bronze layer created in {catalog_name}.{bronze_schema}")
print(f"  \u2022 raw_orders (with ~5% duplicate rows)")
print(f"  \u2022 raw_customers (with numeric nation codes)")

In [0]:
# ============================================================
# SILVER LAYER: Cleaned, validated, and enriched data
# ============================================================
# Silver applies data quality transformations:
# - Deduplication
# - Joining reference data (resolving codes to names)
# - Standardising column names

spark.sql(f"USE SCHEMA `{silver_schema}`")

# Create orders: deduplicated and with clean column names
spark.sql(f"""
CREATE OR REPLACE TABLE orders
COMMENT 'Validated orders - deduplicated from bronze raw_orders'
AS
SELECT DISTINCT
  o_orderkey AS order_id,
  o_custkey AS customer_id,
  o_orderstatus AS order_status,
  o_totalprice AS total_price,
  o_orderdate AS order_date,
  o_orderpriority AS order_priority,
  o_clerk AS clerk,
  o_shippriority AS ship_priority
FROM `{catalog_name}`.`{bronze_schema}`.raw_orders
""")

# Create customers: joined with nation reference to resolve codes to names
spark.sql(f"""
CREATE OR REPLACE TABLE customers
COMMENT 'Enriched customers - nation codes resolved to names, standardised columns'
AS
SELECT 
  c_custkey AS customer_id,
  c_name AS customer_name,
  c_address AS address,
  n.n_name AS nation,
  r.r_name AS region,
  c_phone AS phone,
  c_acctbal AS account_balance,
  c_mktsegment AS market_segment
FROM `{catalog_name}`.`{bronze_schema}`.raw_customers c
JOIN samples.tpch.nation n ON c.c_nationkey = n.n_nationkey
JOIN samples.tpch.region r ON n.n_regionkey = r.r_regionkey
""")

print(f"\u2713 Silver layer created in {catalog_name}.{silver_schema}")
print(f"  \u2022 orders (deduplicated, clean column names)")
print(f"  \u2022 customers (nation codes resolved, region added)")

In [0]:
# ============================================================
# GOLD LAYER: Business-ready, aggregated for dashboards
# ============================================================
# Gold is optimised for specific business questions.
# These tables are ready to be queried directly by dashboards.

spark.sql(f"USE SCHEMA `{gold_schema}`")

# Create daily_revenue: a daily summary for revenue dashboards
spark.sql(f"""
CREATE OR REPLACE TABLE daily_revenue
COMMENT 'Daily revenue summary by nation and segment - ready for dashboard consumption'
AS
SELECT 
  o.order_date,
  c.nation,
  c.region,
  c.market_segment,
  COUNT(o.order_id) AS order_count,
  ROUND(SUM(o.total_price), 2) AS total_revenue,
  ROUND(AVG(o.total_price), 2) AS avg_order_value
FROM `{catalog_name}`.`{silver_schema}`.orders o
JOIN `{catalog_name}`.`{silver_schema}`.customers c ON o.customer_id = c.customer_id
GROUP BY o.order_date, c.nation, c.region, c.market_segment
""")

# Create customer_spending: per-customer lifetime metrics
spark.sql(f"""
CREATE OR REPLACE TABLE customer_spending
COMMENT 'Customer lifetime spending metrics - ready for analytics and segmentation'
AS
SELECT 
  c.customer_id,
  c.customer_name,
  c.nation,
  c.region,
  c.market_segment,
  COUNT(o.order_id) AS lifetime_orders,
  ROUND(SUM(o.total_price), 2) AS lifetime_revenue,
  ROUND(AVG(o.total_price), 2) AS avg_order_value,
  MIN(o.order_date) AS first_order_date,
  MAX(o.order_date) AS last_order_date
FROM `{catalog_name}`.`{silver_schema}`.customers c
JOIN `{catalog_name}`.`{silver_schema}`.orders o ON c.customer_id = o.customer_id
GROUP BY c.customer_id, c.customer_name, c.nation, c.region, c.market_segment
""")

print(f"\u2713 Gold layer created in {catalog_name}.{gold_schema}")
print(f"  \u2022 daily_revenue (aggregated for dashboards)")
print(f"  \u2022 customer_spending (customer lifetime metrics)")

In [0]:
# Add column comments to Gold tables for discoverability
spark.sql(f"""
ALTER TABLE `{catalog_name}`.`{gold_schema}`.daily_revenue ALTER COLUMN order_count COMMENT 'Number of orders placed on this date for this segment';
""")
spark.sql(f"""
ALTER TABLE `{catalog_name}`.`{gold_schema}`.daily_revenue ALTER COLUMN total_revenue COMMENT 'Sum of order values in USD';
""")
spark.sql(f"""
ALTER TABLE `{catalog_name}`.`{gold_schema}`.customer_spending ALTER COLUMN lifetime_orders COMMENT 'Total number of orders placed by this customer';
""")
spark.sql(f"""
ALTER TABLE `{catalog_name}`.`{gold_schema}`.customer_spending ALTER COLUMN lifetime_revenue COMMENT 'Total revenue from this customer in USD';
""")

print("\u2713 Column comments added to Gold tables")

In [0]:
# Set schema context to Gold for the demo notebook
spark.sql(f"USE SCHEMA `{gold_schema}`")

print("")
print("="*60)
print("\u2713 SETUP COMPLETE")
print("="*60)
print(f"")
print(f"Catalog: {catalog_name}")
print(f"")
print(f"Bronze: {bronze_schema}")
print(f"  \u2022 raw_orders      (raw with duplicates)")
print(f"  \u2022 raw_customers   (raw with numeric codes)")
print(f"")
print(f"Silver: {silver_schema}")
print(f"  \u2022 orders          (deduplicated, standardised)")
print(f"  \u2022 customers       (enriched with nation/region names)")
print(f"")
print(f"Gold:   {gold_schema}")
print(f"  \u2022 daily_revenue       (dashboard-ready aggregation)")
print(f"  \u2022 customer_spending   (customer-level metrics)")